In [1]:
# path setup: this notebook now lives in revision_scripts/, while the data paths (data/...) and the
# config / netmedgpt imports are relative to the repository root. Safe to re-run; delete if the
# notebook is moved back to the root.
import os, sys
if os.path.basename(os.getcwd()) == 'revision_scripts':
    os.chdir('..')
for p in (os.getcwd(), os.path.join(os.getcwd(), 'revision_scripts')):
    if p not in sys.path:
        sys.path.insert(0, p)


# Recovery of never-walked edges: NetMedGPT vs walk baselines

Check 3+ of `walk_coverage_analysis.ipynb` showed that NetMedGPT recovers training edges that never appear in the walks. This notebook scores **exactly the same edges** (read from `check3x_edges.csv`) with walk-embedding baselines trained on the **same walks**:
- `node2vec_classifier`: skip-gram on node-only walks + logistic regression (as `node2vec_baseline.py`);
- `node2vec_rel_classifier`: skip-gram on the walks with relation tokens + logistic regression;
- `dreamwalk_samecorpus`: DREAMwalk heterogeneous skip-gram (saved embeddings of `dreamwalk_samecorpus.py`) + XGBoost;
- `dreamwalk_full`: the full DREAMwalk pipeline with its **own** walks (teleport-guided, ATC/MeSH similarity; saved embeddings of `dreamwalk_full.py`) + XGBoost. The edge groups are defined by NetMedGPT's walks, so for this method the comparison is: the same edges, each method with its own walks.

The never-walked edges are removed from the baselines' classifier training pairs, so for every method these edges were never seen in training. Same per-edge filtered ranking and degree matching as check 3+.

If NetMedGPT recovers these edges better than the baselines trained on the same walks, the gain comes from the model, not from the walks.

**Run with the `walk_baselines` kernel** (needs `gensim` and `xgboost`), from the repository root, after check 3+. No GPU needed (used for the logistic regression scoring if available).
Output: `data/new_analysis/coverage/{inference}_seed{seed}/check3x_baselines_edges.csv` and `check3x_baselines_summary.csv`.

In [2]:
INFERENCE = "random_link_split"
SEEDS = [2, 3, 4, 5]
BASELINES = ['node2vec_classifier', 'node2vec_rel_classifier', 'dreamwalk_samecorpus', 'dreamwalk_full']

OUT_DIR = "data/new_analysis/coverage"
# saved DREAMwalk embeddings (dreamwalk_samecorpus.py / dreamwalk_full.py)
DREAMWALK_EMB = {"dreamwalk_samecorpus": "data/new_analysis/dreamwalk_samecorpus/{inference}_seed{seed}/embedding_file.pkl",
                 "dreamwalk_full": "data/new_analysis/dreamwalk_full/{inference}_seed{seed}/embedding_file.pkl"}
# skip-gram settings of node2vec_baseline.py (defaults used for the walk-baseline results)
N2V_PARAMS = dict(vector_size=300, window=4, sg=1, negative=5, min_count=1, epochs=5, workers=30)
PAIR_BUDGET = 500000         # (head, candidate) pairs per scoring call

DEGREE_BINS = [1, 2, 6, 21, 101, 1001, float("inf")]
DEGREE_LABELS = ["1", "2-5", "6-20", "21-100", "101-1000", ">1000"]
CHUNK = 1000000

In [3]:
# graph lookups, walk statistics, coverage and check 3+ ranking of walk_coverage_analysis.ipynb (function cells only)
import pickle
import nbformat
from common import walks_to_sentences, embedding_matrix

FUNCTION_CELLS = ("import os", "# Key helpers", "# Training graph", "# Walk statistics", "import scipy.sparse",
                  "from netmedgpt.model")
loaded = 0
for cell in nbformat.read("revision_scripts/walk_coverage_analysis.ipynb", as_version=4).cells:
    if cell.cell_type == "code" and cell.source.startswith(FUNCTION_CELLS):
        exec(cell.source)
        loaded += 1
assert loaded == len(FUNCTION_CELLS), f"found {loaded} of {len(FUNCTION_CELLS)} function cells"


# score matrix from a baseline pair_score_fn(heads, tails, relation_token) (common.py / baseline_utils.py interface)
def pair_score_matrix_fn(pair_score_fn, z, cand):
    cand_cpu = cand.cpu()

    def fn(hb):
        heads = torch.as_tensor(hb, dtype=torch.long).repeat_interleave(len(cand_cpu))
        tails = cand_cpu.repeat(len(hb))
        s = pair_score_fn(heads, tails, z)
        return torch.as_tensor(s).float().cpu().numpy().reshape(len(hb), len(cand_cpu))
    return fn


# walk baselines trained on the same corpus (node2vec_baseline.py / baseline_utils.py code); the never-walked edges
# (exclude: relation -> pair keys) are removed from the classifier training pairs
def build_baselines(split, g, data, exclude, names, device, seed):
    train = split['train_txgnn']
    keys = pair_key(train['x_index'].to_numpy(), train['y_index'].to_numpy(), g['n_nodes'])
    rel = train['relation'].to_numpy()
    drop = np.zeros(len(train), dtype=bool)
    for relation, ex in exclude.items():
        drop |= (rel == relation) & np.isin(keys, ex)
    split_b = dict(split, train_txgnn=train[~drop])
    print(f"Baselines: {int(drop.sum())} never-walked edges removed from the classifier training pairs")

    scorers = {}
    n2v = [(False, 'node2vec_classifier'), (True, 'node2vec_rel_classifier')]
    if any(name in names for _, name in n2v):
        try:
            from gensim.models import Word2Vec
            from node2vec_baseline import fit_logreg_per_relation
        except ImportError as err:
            print(f"node2vec baselines skipped ({err}); use the walk_baselines environment")
        else:
            for keep_relations, name in n2v:
                if name not in names:
                    continue
                print(f"Training {name}...")
                sentences = walks_to_sentences(data['node2vec_walks'], keep_relations=keep_relations)
                w2v = Word2Vec(sentences, seed=seed, **N2V_PARAMS)
                E, _ = embedding_matrix(w2v.wv, split['vocab_size'], N2V_PARAMS['vector_size'])
                scorers[name] = fit_logreg_per_relation(split_b, E, device, seed)
                del sentences, w2v

    for name, emb_path in DREAMWALK_EMB.items():
        if name not in names:
            continue
        path = emb_path.format(inference=split['inference'], seed=seed)
        try:
            from baseline_utils import fit_xgb_per_relation
            import xgboost  # noqa: F401
        except ImportError as err:
            print(f"{name} skipped ({err}); use the walk_baselines environment")
            continue
        if not os.path.exists(path):
            print(f"{name} skipped: {path} not found")
            continue
        print(f"Training {name} XGBoost classifiers...")
        with open(path, "rb") as f:
            embeddings = pickle.load(f)
        E, _ = embedding_matrix(embeddings, split['vocab_size'], len(next(iter(embeddings.values()))))
        scorers[name] = fit_xgb_per_relation(split_b, E, seed, n_jobs=N2V_PARAMS['workers'])
    return scorers


# never-walked training edges of every queried relation (query direction), as pair keys
def never_walked(split, g, st):
    cov_und = covered_undirected(g, st)
    exclude = {}
    for r, relation in enumerate(split['queried_edge_types']):
        head_type, tail_type = split['queried_node_types'][r].split('|')
        pos = training_positives_by_coverage(split, g, cov_und, relation)
        pos = pos[(g['node_type'][pos['x_index'].to_numpy()] == head_type)
                  & (g['node_type'][pos['y_index'].to_numpy()] == tail_type) & ~pos['seen']]
        exclude[relation] = pair_key(pos['x_index'].to_numpy(), pos['y_index'].to_numpy(), g['n_nodes'])
    return exclude


# scores the edges of check 3+ (NetMedGPT rows of check3x_edges.csv) with every baseline
def score_baselines(split, data, nm_edges, names, device):
    g = build_graph(split)
    st = WalkStats(g)
    walks = data['node2vec_walks'].numpy()
    for i in range(0, len(walks), CHUNK):
        st.add(walks[i:i + CHUNK])
    scorers = build_baselines(split, g, data, never_walked(split, g, st), names, device, split['seed'])

    out = []
    edge = split['edge']
    for r, relation in enumerate(split['queried_edge_types']):
        e = nm_edges[nm_edges['relation'] == relation][['group', 'head', 'tail', 'head_degree', 'tail_degree']]
        e = e.reset_index(drop=True)
        z = int(edge.loc[edge['relation'] == relation, 'z_index'].unique().item())
        cand = split['all_ids_node2'][z]
        known = kg_tails(split, relation, split['queried_node_types'][r])
        for method, pair_fn in scorers.items():
            fn = pair_score_matrix_fn(pair_fn, z, cand)
            auc, hit = edge_ranks(fn, e, cand, known, max(1, PAIR_BUDGET // len(cand)))
            res = e.assign(relation=relation, method=method, auc_edge=auc, hit100=hit)
            out.append(res)
            print(f"{relation} | {method}\n{res.groupby('group', sort=False)[['auc_edge', 'hit100']].mean().round(4).to_string()}")
    return pd.concat(out, ignore_index=True)

## Run

In [4]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

for seed in SEEDS:
    print(f"===== {INFERENCE} | seed {seed} =====")
    run_dir = os.path.join(OUT_DIR, f"{INFERENCE}_seed{seed}")
    nm_edges = pd.read_csv(os.path.join(run_dir, "check3x_edges.csv"))
    nm_edges = nm_edges[nm_edges['method'] == 'NetMedGPT']

    split = load_split(INFERENCE, seed)
    baseline_edges = score_baselines(split, build_corpus(split), nm_edges, BASELINES, device)
    # keep the saved results of baselines that are not in BASELINES (e.g. when only dreamwalk_full is added)
    edges_csv = os.path.join(run_dir, "check3x_baselines_edges.csv")
    if os.path.exists(edges_csv):
        old = pd.read_csv(edges_csv)
        baseline_edges = pd.concat([old[~old['method'].isin(baseline_edges['method'].unique())], baseline_edges],
                                   ignore_index=True)
    baseline_edges.to_csv(edges_csv, index=False)

    both = pd.concat([nm_edges, baseline_edges], ignore_index=True).assign(inference=INFERENCE, seed=seed)
    check3x_summary(both).to_csv(os.path.join(run_dir, "check3x_baselines_summary.csv"), index=False)
    del split

===== random_link_split | seed 2 =====
Reproducibility settings
  seed                     : 2
  torch.manual_seed        : 2
  torch.cuda.manual_seed   : 2
  numpy / random seed      : 2
  cudnn.deterministic      : True
  cudnn.benchmark          : False


/home/bbc8731/NetMedGPT/revision_scripts/common.py:84: DtypeWarning: Columns (1,4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, **cfg['read_kwargs'])
/home/bbc8731/NetMedGPT/revision_scripts/common.py:84: DtypeWarning: Columns (1,4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, **cfg['read_kwargs'])
/home/bbc8731/NetMedGPT/revision_scripts/common.py:84: DtypeWarning: Columns (1,4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, **cfg['read_kwargs'])


Loading cached corpus: data/new_analysis/corpus/random_link_split_seed2.pt
Baselines: 20071 never-walked edges removed from the classifier training pairs
Training node2vec_classifier...


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_fl

Logistic regression trained for indication: 6747 positives, 6747 negatives
Logistic regression trained for contraindication: 17951 positives, 17951 negatives
Logistic regression trained for off-label use: 1973 positives, 1973 negatives
Logistic regression trained for drug_protein: 18874 positives, 18874 negatives
Logistic regression trained for drug_effect: 44874 positives, 44874 negatives
Training node2vec_rel_classifier...


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_fl

... (15 outputs left out)

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_fl

Logistic regression trained for indication: 6780 positives, 6780 negatives
Logistic regression trained for contraindication: 18027 positives, 18027 negatives
Logistic regression trained for off-label use: 1967 positives, 1967 negatives
Logistic regression trained for drug_protein: 18947 positives, 18947 negatives
Logistic regression trained for drug_effect: 44796 positives, 44796 negatives
Training node2vec_rel_classifier...


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_fl

Logistic regression trained for indication: 6780 positives, 6780 negatives
Logistic regression trained for contraindication: 18027 positives, 18027 negatives
Logistic regression trained for off-label use: 1967 positives, 1967 negatives
Logistic regression trained for drug_protein: 18947 positives, 18947 negatives
Logistic regression trained for drug_effect: 44796 positives, 44796 negatives
Training dreamwalk_samecorpus XGBoost classifiers...
XGBoost trained for indication: 6780 positives, 6780 negatives
XGBoost trained for contraindication: 18027 positives, 18027 negatives
XGBoost trained for off-label use: 1967 positives, 1967 negatives
XGBoost trained for drug_protein: 18947 positives, 18947 negatives
XGBoost trained for drug_effect: 44796 positives, 44796 negatives
Training dreamwalk_full XGBoost classifiers...
XGBoost trained for indication: 6780 positives, 6780 negatives
XGBoost trained for contraindication: 18027 positives, 18027 negatives
XGBoost trained for off-label use: 1967 

## Results: per-edge AUROC and Hits@100, degree-matched (same edges for all methods)

In [5]:
from IPython.display import display

res = pd.concat([pd.read_csv(os.path.join(OUT_DIR, f"{INFERENCE}_seed{seed}", "check3x_baselines_summary.csv"))
                 for seed in SEEDS], ignore_index=True)
res['group'] = pd.Categorical(res['group'], categories=GROUP_ORDER, ordered=True)
for v in ['auroc_degree_matched', 'hits100_degree_matched']:
    print(v)
    display(res.pivot_table(index=['relation', 'group'], columns='method', values=v, aggfunc='mean', observed=True).round(3))

auroc_degree_matched


method                                             NetMedGPT  dreamwalk_full  \
relation         group                                                         
contraindication seen                                  0.992           0.985   
                 not adjacent (all)                    0.975           0.901   
                 not adjacent, same walk               0.984           0.910   
                 not adjacent, never in same walk      0.975           0.900   
                 test                                  0.979           0.906   
drug_effect      seen                                  0.998           0.998   
                 not adjacent (all)                    0.998           0.998   
                 not adjacent, same walk               0.998           0.998   
                 not adjacent, never in same walk      0.998           0.998   
                 test                                  0.997           0.996   
drug_protein     seen                                  0.998           0.998   
                 not adjacent (all)                    0.987           0.985   
                 not adjacent, same walk               0.999           0.998   
                 not adjacent, never in same walk      0.986           0.983   
                 test                                  0.993           0.980   
indication       seen                                  0.997           0.996   
                 not adjacent (all)                    0.934           0.944   
                 not adjacent, same walk               0.985           0.965   
                 not adjacent, never in same walk      0.933           0.943   
                 test                                  0.970           0.942   
off-label use    seen                                  0.997           0.999   
                 not adjacent (all)                    0.979           0.908   
                 not adjacent, same walk               0.992           0.925   
                 not adjacent, never in same walk      0.978           0.907   
                 test                                  0.991           0.913   

method                                             dreamwalk_samecorpus  \
relation         group                                                    
contraindication seen                                             0.971   
                 not adjacent (all)                               0.865   
                 not adjacent, same walk                          0.894   
                 not adjacent, never in same walk                 0.864   
                 test                                             0.871   
drug_effect      seen                                             0.997   
                 not adjacent (all)                               0.997   
                 not adjacent, same walk                          0.998   
                 not adjacent, never in same walk                 0.997   
                 test                                             0.995   
drug_protein     seen                                             0.992   
                 not adjacent (all)                               0.941   
                 not adjacent, same walk                          0.990   
                 not adjacent, never in same walk                 0.934   
                 test                                             0.942   
indication       seen                                             0.984   
                 not adjacent (all)                               0.796   
                 not adjacent, same walk                          0.861   
                 not adjacent, never in same walk                 0.794   
                 test                                             0.833   
off-label use    seen                                             0.994   
                 not adjacent (all)                               0.802   
                 not adjacent, same wal

hits100_degree_matched


method                                             NetMedGPT  dreamwalk_full  \
relation         group                                                         
contraindication seen                                  0.832           0.723   
                 not adjacent (all)                    0.582           0.351   
                 not adjacent, same walk               0.649           0.361   
                 not adjacent, never in same walk      0.580           0.351   
                 test                                  0.642           0.380   
drug_effect      seen                                  0.915           0.941   
                 not adjacent (all)                    0.907           0.914   
                 not adjacent, same walk               0.938           0.939   
                 not adjacent, never in same walk      0.899           0.907   
                 test                                  0.883           0.867   
drug_protein     seen                                  0.905           0.880   
                 not adjacent (all)                    0.845           0.786   
                 not adjacent, same walk               0.965           0.934   
                 not adjacent, never in same walk      0.824           0.758   
                 test                                  0.835           0.729   
indication       seen                                  0.955           0.940   
                 not adjacent (all)                    0.683           0.537   
                 not adjacent, same walk               0.787           0.609   
                 not adjacent, never in same walk      0.679           0.533   
                 test                                  0.792           0.588   
off-label use    seen                                  0.958           0.990   
                 not adjacent (all)                    0.754           0.480   
                 not adjacent, same walk               0.846           0.395   
                 not adjacent, never in same walk      0.744           0.484   
                 test                                  0.880           0.561   

method                                             dreamwalk_samecorpus  \
relation         group                                                    
contraindication seen                                             0.544   
                 not adjacent (all)                               0.271   
                 not adjacent, same walk                          0.320   
                 not adjacent, never in same walk                 0.269   
                 test                                             0.306   
drug_effect      seen                                             0.899   
                 not adjacent (all)                               0.874   
                 not adjacent, same walk                          0.913   
                 not adjacent, never in same walk                 0.864   
                 test                                             0.836   
drug_protein     seen                                             0.689   
                 not adjacent (all)                               0.692   
                 not adjacent, same walk                          0.888   
                 not adjacent, never in same walk                 0.657   
                 test                                             0.603   
indication       seen                                             0.652   
                 not adjacent (all)                               0.231   
                 not adjacent, same walk                          0.279   
                 not adjacent, never in same walk                 0.230   
                 test                                             0.298   
off-label use    seen                                             0.887   
                 not adjacent (all)                               0.300   
                 not adjacent, same wal